# NB3｜峰位判讀：食品分子的指紋

**食品分析｜拉曼光譜與 RamanSPy 入門系列（第 3 本，共 5 本）**

會跑程式還不夠 —— 食品分析師真正的價值，是**看懂峰位代表什麼分子**。這一本把光譜和化學連起來。

---
### 這一本你會學到
- 用 `rp.plot.peaks()` 自動找出峰位
- 對照食品成分的特徵峰表
- 理解「強度 ≠ 含量」這個常見誤解
- 判讀一個未知樣品

> 💡 **完全沒寫過程式也沒關係。** 你只要做三件事：
> 1. 用滑鼠點每一格左邊的 ▶ 播放鍵（或按 `Shift + Enter`）
> 2. 看下面跑出來的圖和數字
> 3. 遇到 `# 👉 換你做` 的地方，照提示改一個數字或一個字，再跑一次


In [ ]:
# ===== 第一次執行請先跑這一格（大約 1 分鐘）=====
# 在 Google Colab 上，套件不是永久安裝的，每次重開都要跑一次。
!pip install -q ramanspy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ramanspy as rp

# 讓圖上的中文正常顯示（Colab 用）
!wget -q -O TaipeiSans.ttf https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_ 2>/dev/null
import matplotlib
try:
    matplotlib.font_manager.fontManager.addfont("TaipeiSans.ttf")
    matplotlib.rc("font", family="Taipei Sans TC Beta")
except Exception:
    pass
matplotlib.rcParams["axes.unicode_minus"] = False

print("準備完成！")


In [ ]:
# ===== 資料載入設定 =====
# 這一行由老師部署時自動填入正確的 GitHub 網址，學生不用改。
DATA_BASE = "https://raw.githubusercontent.com/Tai-ShengYeh/Tai-ShengYeh.github.io/main/ramanspy-food-analysis/data/"

# 若你把 CSV 直接上傳到 Colab 左側「檔案」，把上面那行改成： DATA_BASE = ""
# 若你在自己電腦跑，且 data 資料夾就在旁邊，改成：       DATA_BASE = "data/"

def load_spectra(filename):
    """讀 CSV → 回傳 (樣品資訊表 meta, ramanspy 光譜物件 spectra)"""
    df = pd.read_csv(DATA_BASE + filename)
    meta_cols = [c for c in df.columns if not c.replace(".", "", 1).isdigit()]
    axis = np.array([float(c) for c in df.columns if c not in meta_cols])
    spectra = rp.SpectralContainer(df.drop(columns=meta_cols).values, axis)
    return df[meta_cols].reset_index(drop=True), spectra

print("load_spectra() 已定義，資料來源：", DATA_BASE or "（Colab 本機檔案）")


In [ ]:
# 本課程統一使用的標準前處理流程
pipeline = rp.preprocessing.Pipeline([
    rp.preprocessing.misc.Cropper(region=(450, 1800)),          # 裁切
    rp.preprocessing.despike.WhitakerHayes(),                    # 去宇宙射線
    rp.preprocessing.denoise.SavGol(window_length=9, polyorder=3),  # 平滑
    rp.preprocessing.baseline.IModPoly(),                        # 基線校正
    rp.preprocessing.normalise.MinMax(),                         # 歸一化
])


## 1. 六種食品成分的純物質光譜

In [ ]:
ref = pd.read_csv(DATA_BASE + "reference_spectra.csv")
rx = ref["raman_shift_cm-1"].values

names = {"lactose": "乳糖", "casein": "酪蛋白", "melamine": "三聚氰胺",
         "starch": "澱粉", "tg_base": "油脂骨架", "carotene": "類胡蘿蔔素"}

plt.figure(figsize=(9, 6))
for i, (k, label) in enumerate(names.items()):
    plt.plot(rx, ref[k] / ref[k].max() + i * 1.2, lw=1.1)
    plt.text(1810, i * 1.2 + 0.3, label, fontsize=10)
plt.xlim(400, 1800); plt.yticks([])
plt.xlabel("拉曼位移 (cm$^{-1}$)")
plt.title("食品常見成分的拉曼指紋")
plt.show()

## 2. 食品分析必背的特徵峰對照表

| 波數 (cm⁻¹) | 歸屬 | 在食品裡代表 |
|---|---|---|
| 478 | C–C–O / 環變形 | **澱粉**（診斷峰）|
| 676 | 三嗪環呼吸 | **三聚氰胺**（摻偽物診斷峰）|
| 850 / 1085 / 1125 | C–C、C–O 伸縮 | 醣類（乳糖、蔗糖）|
| 1003 | 苯環呼吸（苯丙胺酸）| **蛋白質**（強度穩定，常當內標）|
| 1156 / 1523 | C–C（ν₂）/ C=C（ν₁）共軛鏈 | **類胡蘿蔔素**（共振增強，超強）|
| 1265 | =C–H 面內變形（順式）| 油脂**不飽和度** |
| 1301 | CH₂ 扭曲 | 油脂飽和鏈長 |
| 1441 | CH₂ 剪式變形 | 油脂總量（常當內標）|
| 1655 | C=C 伸縮 / 醯胺 I | 不飽和脂肪 **或** 蛋白質二級結構 |
| 1745 | C=O 伸縮（酯）| 三酸甘油酯 |

> ⚠️ **1655 是陷阱題**：在油脂樣品裡它是 C=C；在蛋白質樣品裡它是醯胺 I。**必須先知道基質是什麼**才能判讀。

## 3. 讓程式自動找峰

`rp.plot.peaks()` 用 `prominence`（突出度）決定「多凸才算一個峰」。

In [ ]:
meta, spectra = load_spectra("milk_powder_melamine.csv")
processed = pipeline.apply(spectra)

# 挑一個摻了 5% 三聚氰胺的樣品
idx = int(meta.melamine_pct.idxmax())
one = rp.Spectrum(processed.spectral_data[idx], processed.spectral_axis)

ax, peaks, props = rp.plot.peaks(one, prominence=0.05, return_peaks=True)
rp.plot.show()
print("找到的峰位 (cm-1)：", peaks)

### 👉 換你做

把 `PROM` 改成 0.01、0.02、0.15、0.3，觀察找到的峰數量。

**思考**：`prominence` 太小會怎樣？太大會怎樣？

In [ ]:
PROM = 0.05     # 👉 換你做

ax, peaks, props = rp.plot.peaks(one, prominence=PROM, return_peaks=True)
rp.plot.show()
print(f"prominence={PROM} → 找到 {len(peaks)} 個峰：{peaks}")

## 4. 重要觀念：強度 ≠ 含量

類胡蘿蔔素在橄欖油裡只佔 **百萬分之幾**，但它的 1523 cm⁻¹ 峰卻是整張圖最強的。為什麼？

因為 532 nm 綠光雷射剛好落在類胡蘿蔔素的電子吸收帶內，產生**共振拉曼增強（resonance Raman scattering）**。
文獻上類胡蘿蔔素的共振增強倍率**約為五個數量級（~10⁵）**，最高可達六個數量級。

**實務推論**：
- 峰很強 ≠ 含量很多 → **不能用峰高直接當濃度**
- 想定量一定要**做檢量線**（NB5 會做）
- 強散射體會**遮蔽**弱散射體 → 主成分的峰可能把摻偽物的峰蓋掉

In [ ]:
# 用油品資料看共振增強的威力
ometa, ospectra = load_spectra("edible_oils.csv")
oproc = pipeline.apply(ospectra)

plt.figure(figsize=(9, 3.5))
for k, label in [("olive", "橄欖油（含類胡蘿蔔素）"), ("coconut", "椰子油（幾乎不含）")]:
    m = (ometa.oil_type == k).values
    plt.plot(oproc.spectral_axis, oproc.spectral_data[m].mean(0), label=label)
plt.axvline(1523, ls=":", color="r"); plt.text(1530, 0.6, "1523 cm$^{-1}$", color="r")
plt.legend(); plt.xlabel("拉曼位移 (cm$^{-1}$)"); plt.title("共振增強：含量極少，訊號極強")
plt.show()

## 5. 實戰：判讀未知樣品

下面有 6 個未知樣品。跑出來後，用上面的對照表推理它們是什麼。

In [ ]:
umeta, uspectra = load_spectra("unknown_samples.csv")
uproc = pipeline.apply(uspectra)

fig, axes = plt.subplots(3, 2, figsize=(11, 8))
for i, ax in enumerate(axes.ravel()):
    ax.plot(uproc.spectral_axis, uproc.spectral_data[i], lw=.9)
    ax.set_title(umeta.sample_id[i])
    for w in [478, 676, 1003, 1085, 1441, 1523, 1745]:
        ax.axvline(w, ls=":", lw=.6, color="grey")
plt.tight_layout(); plt.show()

In [ ]:
# 幫你把幾個關鍵波數的強度列成表，方便判讀
key = [478, 676, 1003, 1085, 1441, 1523, 1656, 1745]
tab = pd.DataFrame({f"{w}": uproc.spectral_data[:, np.argmin(abs(uproc.spectral_axis - w))].round(3)
                    for w in key})
tab.insert(0, "sample", umeta.sample_id)
tab

### 🧪 自我檢核

1. 某樣品在 478 cm⁻¹ 有強峰、1745 cm⁻¹ 幾乎沒有訊號 —— 最可能是什麼？
2. 某奶粉樣品在 676 cm⁻¹ 出現明顯峰 —— 代表什麼？可以直接下結論說「摻了 5%」嗎？
3. 兩個油品樣品，A 的 1265 / 1441 比值比 B 高，哪一個比較不飽和？
4. 為什麼「1523 cm⁻¹ 峰很強」不能推論「類胡蘿蔔素含量很高」？

<details><summary>▶ 點開看參考答案</summary>

1. 澱粉類（478 是澱粉診斷峰，1745 酯基缺席表示不是油脂）。
2. 代表**檢出三聚氰胺**（定性）。但不能直接說濃度 —— 定量必須先用已知濃度的標準品建立檢量線，並確認在線性範圍內。
3. A。1265 cm⁻¹ 對應順式 =C–H，比值越高不飽和度越高；1441（CH₂）當內標。
4. 因為共振拉曼增強。訊號強度同時取決於「含量」和「散射截面」，共振物種的散射截面可以大好幾個數量級。

</details>


---
### 📚 這一本用到的資料與文獻

**資料**：`data/` 內的光譜為**依文獻峰位建立的模擬資料**（`make_data.py`，亂數種子 20260801），刻意加入螢光背景、宇宙射線與雜訊。可用於教學演練，**不可引用為實驗證據**。

**主要文獻**

- Georgiev, D. et al. *RamanSPy: An Open-Source Python Package for Integrative Raman Spectroscopy Data Analysis*. **Anal. Chem.** 2024, 96(21), 8492–8500. doi:10.1021/acs.analchem.4c00383
- Gill, D.; Kilponen, R. G.; Rimai, L. *Resonance Raman Scattering … in Intact Plant Tissues*. **Nature** 1970, 227, 743–744. doi:10.1038/227743a0
- Lu, L. et al. *Resonance Raman scattering of β-carotene … second singlet state*. **J. Photochem. Photobiol. B** 2018, 179, 18–22. doi:10.1016/j.jphotobiol.2017.12.022
- Withnall, R. et al. *Raman spectra of carotenoids in natural products*. **Spectrochim. Acta A** 2003, 59(10), 2207–2212. doi:10.1016/S1386-1425(03)00064-7
- de Oliveira, V. E. et al. *Carotenes and carotenoids in natural biological samples*. **J. Raman Spectrosc.** 2010, 41(6), 642–650. doi:10.1002/jrs.2493
- Portarena, S. et al. *Cultivar discrimination, fatty acid profile and carotenoid characterization of monovarietal olive oils by Raman spectroscopy at a single glance*. **Food Control** 2019, 96, 137–145. doi:10.1016/j.foodcont.2018.09.011
- Chen, Y. et al. *Quantitative analysis of β-carotene and unsaturated fatty acids in blended olive oil via Raman spectroscopy combined with model prediction*. **Food Chemistry** 2025, 470, 142621. doi:10.1016/j.foodchem.2024.142621
- Schmidt, W. et al. *Continuous Temperature-Dependent Raman Spectroscopy of Melamine and Structural Analog Detection in Milk Powder*. **Appl. Spectrosc.** 2015, 69(3), 398–406. doi:10.1366/14-07600
- Zhang, X. et al. *Detection of melamine in liquid milk using SERS*. **J. Raman Spectrosc.** 2010, 41(12), 1655–1660. doi:10.1002/jrs.2629
- Kim, A. et al. *Melamine Sensing in Milk Products by Using SERS*. **Anal. Chem.** 2012, 84(21), 9303–9309. doi:10.1021/ac302025q
- FAO/WHO Codex Alimentarius. *General Standard for Contaminants and Toxins in Food and Feed*, **CXS 193-1995**.

完整清單見課程網站的「數據來源」與「參考文獻」兩節。